### Building models with the `nn.Module`

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"

#### Tiny Model

`nn.Module`:
1. Is the base class for **all neural network** modules in PyTorch
2. Provides automatic gradient computation, parameter management, and device placement
3. Need to call `super().__init__()` in the constructor to initialize the base class

In [2]:

class TinyNet(nn.Module):
    def __init__(self, in_features=4, hidden=8, out_features=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden), # 4 → 8
            nn.ReLU(),                      # Activation function
            nn.Linear(hidden, out_features) # 8 → 1
        )
    def forward(self, x):
        return self.net(x)


##### Network Architecture
`nn.Linear(4, 8)`: Fully connected layer
1. Input: 4 features
2. Output: 8 neurons
3. Parameters: 4×8 + 8 = 40 (weights + biases)

`nn.ReLU()`: Activation function
1. Applies max(0, x) element-wise
2. Introduces non-linearity
3. No learnable parameters

`nn.Linear(8, 1)`: Output layer
1. Input: 8 features
2. Output: 1 value
3. Parameters: 8×1 + 1 = 9 (weights + biases)

**Total parameters**: 40 + 9 = 49

##### The Forward Pass
1. **Required method**: All `nn.Module` subclasses must implement `forward()`
2. **Flow**: Input → Linear → ReLU → Linear → Output
3. **Automatic**: PyTorch automatically builds the computational graph for backpropagation

In [3]:
model = TinyNet().to(device) # Moves model to GPU if available, otherwise CPU
criterion = nn.MSELoss() # Mean Squared Error: (prediction - target)²
optimizer = optim.Adam(model.parameters(), lr=1e-2)

##### The Optimizer
1. **Adam optimizer**: Adaptive learning rate algorithm
2. `model.parameters()`: Gets all trainable parameters (weights & biases)
3. **lr=1e-2**: Learning rate of 0.01
4. **Purpose**: Updates parameters to minimize loss

### Training models (the 5 steps)
1. zero the gradients
2. forward pass
3. compute loss
4. backward pass
5. optimizer step

In [5]:
for epoch in range(100):
    optimizer.zero_grad()                 # (1) zero the gradients (clear old gradients)

    # Different batches for each epoch
    x_batch = torch.randn(32, 4).to(device)
    y_batch = torch.randn(32, 1).to(device)

    # PyTorch automatically provides something like:
    # def __call__(self, x):
    #     return self.forward(x)
    # The __call__ method ensures proper PyTorch behavior
    preds = model(x_batch)                # (2) forward pass, compute predictions
    


    loss = criterion(preds, y_batch)      # (3) compute the loss, criterion = nn.MSELoss()
    
    
    loss.backward()                       # (4) backward pass (gradient calcualtion)
    
    # Without optimizer.step(), your model would never learn
    # the weights would stay the same forever!
    # It's the crucial step that makes the network get better at its task.    
    optimizer.step()                      # (5) Updates all model parameters using gradients
                                          # (5) parameter = parameter - learning_rate * gradient



    if epoch % 10 == 0:
        print(f"epoch {epoch:03d}  loss {loss.item():.4f}")


epoch 000  loss 0.8277
epoch 010  loss 1.1182
epoch 020  loss 0.7903
epoch 030  loss 1.0911
epoch 040  loss 0.9677
epoch 050  loss 1.0105
epoch 060  loss 0.6707
epoch 070  loss 1.5928
epoch 080  loss 0.9805
epoch 090  loss 0.7438


##### Why use `model(x)` instead of `model.forward(x)`?
1. Hooks and preprocessing: `__call__` handles important PyTorch functionality:
- Registering forward hooks
- Setting training/eval mode properly
- Gradient computation setup
2. Cleaner syntax: model(x_batch) is more intuitive than model.forward(x_batch)
3. Convention: This is the standard PyTorch pattern - everyone does it this way

### Parameter Inspection

In [6]:
# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Inspect layers
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")

# Model summary
print(model)

net.0.weight: torch.Size([8, 4])
net.0.bias: torch.Size([8])
net.2.weight: torch.Size([1, 8])
net.2.bias: torch.Size([1])
TinyNet(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=1, bias=True)
  )
)


### Evaluation

In [7]:
# Standard PyTorch inference pattern
model.eval()                    # Switch to evaluation mode
with torch.no_grad():          # Disable gradient computation
    test_x = torch.randn(5, 4).to(device)
    predictions = model(test_x)  # Get predictions
    print(predictions)

tensor([[ 0.1244],
        [ 0.1247],
        [ 0.0556],
        [-0.0418],
        [ 0.1141]])


##### `model.eval()` and `torch.no_grad()`
1. `model.eval()`: Switches the model to evaluation mode
- Disables dropout and batch normalization updates
2. `with torch.no_grad()`: Context manager that disables gradient computation
- Reduces memory consumption
- Speeds up computations during inference

### Model State and Checkpoints

In [8]:
# Saving
torch.save(model.state_dict(), 'model.pth')


# To load:
model = TinyNet()  # Create model instance first
model.load_state_dict(torch.load('model.pth'))
model.eval()  # Set to evaluation mode


TinyNet(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=1, bias=True)
  )
)

**Advantages**:
1. *Only saves parameters*: Weights and biases, not the class structure
2. *Portable*: Works across different scripts and environments
3. *Robust*: Less likely to break between PyTorch versions
4. *Smaller files*: Only stores the actual learned parameters

In [9]:
# Save with additional metadata
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': epoch,
    'loss': loss.item(),
    'model_config': {
        'in_features': 4,
        'hidden': 8,
        'out_features': 1
    }
}
torch.save(checkpoint, 'checkpoint.pth')

# Load with error handling
def load_checkpoint(filepath):
    try:
        checkpoint = torch.load(filepath)
        model = TinyNet().to(device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        return model, checkpoint
    except FileNotFoundError:
        print(f"Checkpoint {filepath} not found")
        return None, None

# Usage
loaded_model , checkpoints = load_checkpoint('checkpoint.pth')

In [10]:
loaded_model

TinyNet(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=1, bias=True)
  )
)

In [11]:
checkpoints['model_state_dict']

OrderedDict([('net.0.weight',
              tensor([[-0.0021,  0.3430, -0.2441, -0.3102],
                      [ 0.3847,  0.2196, -0.0868,  0.2594],
                      [ 0.1006, -0.4948,  0.2973, -0.3894],
                      [-0.3217, -0.4941,  0.0024, -0.2092],
                      [-0.0324,  0.2417,  0.3279, -0.3245],
                      [-0.1123,  0.0676,  0.3769,  0.2945],
                      [ 0.3055,  0.1270,  0.1608,  0.3995],
                      [-0.1300,  0.1275,  0.2346, -0.3791]])),
             ('net.0.bias',
              tensor([-0.4342,  0.1845, -0.1736, -0.1699,  0.2929,  0.1102,  0.4018,  0.1448])),
             ('net.2.weight',
              tensor([[-0.1365,  0.0975,  0.0496,  0.0282,  0.0286,  0.1056, -0.1831,  0.0274]])),
             ('net.2.bias', tensor([0.0647]))])

In [12]:
checkpoint['optimizer_state_dict']

{'state': {0: {'step': tensor(200.),
   'exp_avg': tensor([[ 8.4637e-03, -6.7107e-03,  7.6134e-03,  1.2906e-02],
           [ 2.7730e-03,  8.5753e-06, -7.2567e-04, -7.7247e-03],
           [-1.8306e-03, -1.3214e-03, -9.9542e-04, -4.3787e-03],
           [-4.1078e-03, -1.7028e-03, -3.8097e-03, -3.2883e-03],
           [-3.3738e-03,  1.3625e-03, -1.2570e-03, -3.8752e-03],
           [-9.7207e-04,  9.3634e-04,  4.7343e-03, -9.8114e-03],
           [-4.2697e-04,  4.9528e-03, -2.0205e-03,  8.4925e-03],
           [-2.2235e-03,  4.4470e-04, -9.3128e-04, -4.3624e-03]]),
   'exp_avg_sq': tensor([[1.8016e-04, 2.2900e-04, 2.9457e-04, 2.5298e-04],
           [3.2119e-04, 3.7912e-04, 3.8909e-04, 4.2794e-04],
           [1.3474e-05, 2.0909e-05, 1.4991e-05, 1.6527e-05],
           [1.7168e-05, 1.8291e-05, 1.2676e-05, 1.4549e-05],
           [8.9970e-06, 1.3992e-05, 1.2782e-05, 1.5029e-05],
           [5.9383e-05, 5.9438e-05, 6.0134e-05, 8.5791e-05],
           [6.4832e-04, 7.8344e-04, 6.6338e-04, 6.